# Notebook 04: Model Evaluation

Covers: Metrics computation, Confusion Matrix, ROC Curves, and Classification Report.

**Run after Notebook 03 (model must be trained).**


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import Config
from src.utils.helpers import ensure_dirs, save_json

ensure_dirs(*Config.all_output_dirs())
print('Environment ready.')


## Step 1: Load Best Model


In [ ]:
from src.training.trainer import load_best_model

print(f'Loading model from: {Config.BEST_MODEL_PATH}')
model = load_best_model()
print(f'Model loaded. Parameters: {model.count_params():,}')


## Step 2: Build Test Dataset


In [ ]:
from src.preprocessing.data_loader import build_tf_dataset

test_ds = build_tf_dataset(Config.TEST_DIR, split='test')
print(f'Test dataset: {len(test_ds)} batches.')


## Step 3: Compute All Metrics


In [ ]:
from src.evaluation.metrics import compute_metrics, classification_report_df

print('Computing evaluation metrics...')
metrics = compute_metrics(model, test_ds, class_names=Config.CLASS_NAMES)

scalar_metrics = {k: v for k, v in metrics.items() if isinstance(v, float)}
print('\nEVALUATION RESULTS:')
for name, value in scalar_metrics.items():
    print(f'  {name:<30} {value:.4f}')


## Step 4: Classification Report


In [ ]:
print(metrics['report_str'])
report_df = classification_report_df(metrics, Config.CLASS_NAMES)
print('\nDataFrame:')
report_df


## Step 5: Confusion Matrix

- **Normalized**: fraction per true class (shows per-class error rates)
- **Raw counts**: actual sample counts


In [ ]:
from src.evaluation.visualizer import plot_confusion_matrix

fig_norm = plot_confusion_matrix(
    metrics, normalize=True,
    save_path=Config.CONFUSION_MATRIX_DIR / 'confusion_matrix_normalized.png',
    title='Normalized Confusion Matrix',
)
plt.show()

fig_counts = plot_confusion_matrix(
    metrics, normalize=False,
    save_path=Config.CONFUSION_MATRIX_DIR / 'confusion_matrix_counts.png',
    title='Confusion Matrix - Sample Counts',
)
plt.show()


## Step 6: ROC Curves

Multi-class ROC using **One-vs-Rest (OvR)** binarization. Each class has its own AUC score; macro-average is the summary.


In [ ]:
from src.evaluation.visualizer import plot_roc_curve

fig_roc = plot_roc_curve(
    metrics,
    save_path=Config.PLOTS_DIR / 'roc_curves.png',
    title='Multi-Class ROC Curves (One-vs-Rest)',
)
plt.show()


## Step 7: Save All Reports


In [ ]:
from src.evaluation.visualizer import save_classification_report

save_classification_report(
    metrics,
    csv_path=Config.METRICS_DIR / 'classification_report.csv',
    json_path=Config.METRICS_DIR / 'classification_report.json',
    txt_path=Config.METRICS_DIR / 'classification_report.txt',
)
save_json(scalar_metrics, Config.METRICS_DIR / 'scalar_metrics.json')
print('All reports saved!')
print('Next -> Open 05_Explainable_AI_GradCAM.ipynb')
